## Verilerin Lineer Regresyon ile Tahmin Edilmesi

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv('data_cleaned.csv')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6116 entries, 0 to 6115
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   city          6116 non-null   object
 1   district      6116 non-null   object
 2   neighborhood  6116 non-null   object
 3   room          6116 non-null   int64 
 4   living_room   6116 non-null   int64 
 5   area          6116 non-null   int64 
 6   age           6116 non-null   int64 
 7   floor         6116 non-null   int64 
 8   price         6116 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 430.2+ KB
None


In [3]:
categorical_features = ['city', 'district', 'neighborhood']
numerical_features = ['room', 'living_room', 'area', 'age', 'floor']

In [10]:
full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # eğitimde hiç bulunmayan mahalleler testte varsa onları görmezden gelmeyi sağlar. 
])

In [11]:
X = df.drop('price', axis=1) #matrisler için büyük harf
y = df['price'] #vektörler için küçük harf

random_state, aynı seti tekrar yakalamak için gereken tohum-seed. 42 verince aynısını elde edersiniz.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
model = Pipeline([
    ('preparation', full_pipeline),
    ('model', LinearRegression())
])

In [14]:
model.fit(X_train, y_train)

,steps,"[('preparation', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) #root mean squared error
r2 = r2_score(y_test, y_pred)

r^2 0.5 in altında kalırsa model aşırı başarısız. yani gidip herhangi bi ortalamayı kullansa bile bizim modelden başarılı oluyo yani baya amelelik yapmış oluyoz.

In [19]:
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R^2: {r2}")

MSE: 43046419.624643065
RMSE: 6560.977032778203
R^2: 0.6161093979190568


In [21]:
feature_importances = model.named_steps['model'].coef_
print(len(feature_importances)) # parametrelerin toplam adedi
print(feature_importances)

726
[ 8.63088970e+02  0.00000000e+00  3.23401539e+03 -2.15389910e+03
  1.71770522e+02 -3.66625221e+03  1.42255126e+03 -3.32687023e+03
  5.27398922e+03 -2.39757631e+03  2.69415826e+03 -4.35579102e+02
 -1.53891037e+03 -1.18207940e+03  2.25636258e+02  7.93839387e+03
 -9.48512975e+03 -3.86111269e+03 -6.93423769e+03  1.64587900e+04
 -2.39843620e+03 -2.64336735e+03 -2.48739660e+03 -2.63414581e+03
  3.52502529e+03 -2.40913688e+03  3.17152096e+02  1.95307517e+04
 -4.87302889e+03 -3.36970103e+03 -2.31802403e+03 -8.29213332e+02
  1.03196514e+03 -3.47587100e+03 -2.20381466e+03 -3.40835435e+03
  2.90430585e+03  1.06011478e+04 -1.47352808e+03  1.26587494e+04
  2.85775368e+03 -6.68982770e+03 -1.87878005e+03 -7.36534724e+02
  1.44153677e+03 -3.64567616e+03 -4.33036040e+03  4.15916251e+03
 -7.28035351e+02  1.38433470e+04  7.24807668e+03  2.48082748e+02
 -4.26159430e+03  1.84311530e+03 -1.58496811e+03  1.09678861e+03
 -4.23781015e+03  9.46773387e+03 -4.71886196e+03 -8.35582615e+03
 -4.79260877e+03 -9.6

önem katsayıları

In [23]:
print("Numerical Features")
for i in range(len(numerical_features)):
    print(numerical_features[i], feature_importances[i])

Numerical Features
room 863.0889704167782
living_room 0.0
area 3234.0153931007876
age -2153.899097101616
floor 171.77052196232594


In [28]:
print("Categorical Features")
for i in range(len(categorical_features)):
    for j in range(len(model.named_steps['preparation'].transformers_[1][1].categories_[i])):
        print(model.named_steps['preparation'].transformers_[1][1].categories_[i][j], feature_importances[len(numerical_features) + j])

Categorical Features
afyonkarahisar -3666.252206678124
aydin 1422.5512638406162
denizli -3326.870225177088
izmir 5273.9892166013815
manisa -2397.57631010783
mugla 2694.1582615412517
acipayam -3666.252206678124
akhisar 1422.5512638406162
alasehir -3326.870225177088
aliaga 5273.9892166013815
balcova -2397.57631010783
bayindir 2694.1582615412517
bayrakli -435.5791020298581
bergama -1538.9103707860959
bodrum -1182.0794004489119
bolvadin 225.6362575909265
bornova 7938.393866635529
buca -9485.129746690802
buharkent -3861.112694745263
cameli -6934.237693134643
cardak 16458.79000923249
cay -2398.436197213908
cesme -2643.3673469041673
cigli -2487.396598196446
cine -2634.145813195067
dalaman 3525.025294053463
datca -2409.1368791325826
demirci 317.1520955854191
didim 19530.75168397017
dikili -4873.028887128969
efeler -3369.701034763008
fethiye -2318.0240286544104
foca -829.2133323197901
gaziemir 1031.9651409134847
germencik -3475.871004153058
guzelbahce -2203.81466149421
incirliova -3408.35434913

In [31]:
new_data = pd.DataFrame({
    'city': ['manisa'],
    'district': ['yunusemre'],
    'neighborhood': ['guzelyurt'],
    'room': [3],
    'living_room': [1],
    'area': [120],
    'age': [5],
    'floor': [3]
})

print(model.predict(new_data))

[22327.4370072]


In [35]:
print(df[(df['city'] == 'manisa') & (df['district'] == 'yunusemre') & (df['neighborhood'] == 'guzelyurt')])

        city   district neighborhood  room  living_room  area  age  floor  \
4112  manisa  yunusemre    guzelyurt     1            1    65   13      5   
4159  manisa  yunusemre    guzelyurt     2            1    85    2      3   
4183  manisa  yunusemre    guzelyurt     4            1   196    5      1   
4200  manisa  yunusemre    guzelyurt     1            1    60   11      5   

      price  
4112  15000  
4159  15000  
4183  36000  
4200  11000  
